# Reinforcement Learning — Lab 3
## From Tables to Features: Mountain Car with Q-learning and Linear SARSA

**Goal:** learn how prediction and control change when the state is continuous. You will:

1. use **semi-gradient TD(0)** to predict values under a fixed policy;
2. turn a continuous problem into a table using **discretisation**;
3. implement **linear SARSA**; and
4. compare simple state features with **tile coding**.

The central question is: **what does a representation allow an agent to learn?**

### Marking / submission (suggested)

- Your completed notebook (`.ipynb`)
- Completed code in cells marked **TODO**
- Plots comparing the three control agents
- Brief answers to the questions in **Part 7**

---
## Part 1 — Setup (Google Colab)

Run the next cell to install and import the required packages. If you are working locally, you may already have them installed.

In [ ]:
# Install Gymnasium and the classic-control environments.
%pip -q install "gymnasium[classic-control]"

import random
from dataclasses import dataclass
from typing import Callable

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

---
## Part 2 — Meet Mountain Car

A car is trapped in a valley. Its engine is not powerful enough to drive straight up the right-hand hill, so it must build momentum by moving backwards and forwards.

The state is continuous:

$$s=(\text{position},\text{velocity}).$$

The three actions are `0` (push left), `1` (no push), and `2` (push right). The agent receives reward $-1$ on every step, so a better policy reaches the goal in fewer steps. An episode ends at the goal or at the environment's time limit.

In [ ]:
env = gym.make("MountainCar-v0")
state, info = env.reset(seed=SEED)

print("Initial state [position, velocity]:", state)
print("State lower bounds:", env.observation_space.low)
print("State upper bounds:", env.observation_space.high)
print("Number of actions:", env.action_space.n)
print("Maximum episode length:", env.spec.max_episode_steps)

env.close()

### 2.1 A random trajectory

Run one episode with random actions. Notice that return is the negative of the episode length. Reaching the goal sooner therefore gives a larger (less negative) return.

In [ ]:
env = gym.make("MountainCar-v0")
state, info = env.reset(seed=SEED)
trajectory = [state.copy()]
total_reward = 0.0

while True:
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    trajectory.append(state.copy())
    total_reward += reward
    if terminated or truncated:
        break

trajectory = np.asarray(trajectory)
print(f"Return: {total_reward:.0f}; steps: {len(trajectory) - 1}; reached goal: {terminated}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(trajectory[:, 0], label="position")
axes[0].plot(trajectory[:, 1], label="velocity")
axes[0].set(xlabel="step", title="State through time")
axes[0].legend()
axes[1].plot(trajectory[:, 0], trajectory[:, 1])
axes[1].set(xlabel="position", ylabel="velocity", title="Trajectory through state space")
plt.tight_layout()
env.close()

---
## Part 3 — Prediction with semi-gradient TD(0)

Prediction asks: **how good are states when the policy is fixed?** It does not improve the policy.

We will evaluate a supplied heuristic policy that pushes in the direction the car is already travelling. Instead of storing one value per state, approximate the value with

$$\hat v(s,\mathbf w)=\mathbf w^\top\mathbf x(s).$$

For a linear approximator, $\nabla \hat v(s,\mathbf w)=\mathbf x(s)$. The semi-gradient TD(0) update is

$$\delta_t=R_{t+1}+\gamma\hat v(S_{t+1},\mathbf w)-\hat v(S_t,\mathbf w),$$

$$\mathbf w\leftarrow\mathbf w+\alpha\delta_t\mathbf x(S_t).$$

For a terminal transition, the value of the next state is defined to be zero.

In [ ]:
def scale_state(state: np.ndarray, low: np.ndarray, high: np.ndarray) -> np.ndarray:
    """Scale each state component to approximately [-1, 1]."""
    return 2.0 * (np.asarray(state) - low) / (high - low) - 1.0


def polynomial_value_features(state: np.ndarray, low: np.ndarray, high: np.ndarray) -> np.ndarray:
    """A feature vector; the approximation remains linear in its weights."""
    position, velocity = scale_state(state, low, high)
    return np.array([
        1.0, position, velocity, position**2, position * velocity, velocity**2
    ])


def momentum_policy(state: np.ndarray) -> int:
    """Push with the current velocity; initially push right."""
    return 2 if state[1] >= 0 else 0

### 3.1 Implement the TD prediction update

Complete the three TODOs below. `done` includes both natural termination and the time limit, so there is no bootstrap term when it is true.

In [ ]:
def train_linear_td_prediction(
    episodes: int = 500,
    alpha: float = 0.01,
    gamma: float = 1.0,
    seed: int = SEED,
):
    env = gym.make("MountainCar-v0")
    low, high = env.observation_space.low, env.observation_space.high
    w = np.zeros(6, dtype=np.float64)
    returns = []

    for episode in range(episodes):
        state, info = env.reset(seed=seed + episode)
        episode_return = 0.0

        while True:
            action = momentum_policy(state)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            x = polynomial_value_features(state, low, high)
            value = w @ x

            # TODO 1: compute next_value. It must be 0 when done.
            next_value = ...
            # TODO 2: compute the TD error.
            delta = ...
            # TODO 3: update every weight using the semi-gradient.
            w += ...

            episode_return += reward
            state = next_state
            if done:
                break

        returns.append(episode_return)

    env.close()
    return w, np.asarray(returns)

In [ ]:
# Run after completing the TODOs above.
prediction_w, prediction_returns = train_linear_td_prediction()
print("Learned prediction weights:", np.round(prediction_w, 3))
print("Mean policy return over final 50 episodes:", prediction_returns[-50:].mean())

### 3.2 Visualise the predicted value function

The plot is an approximation learned from the states visited by the fixed policy. Be cautious when interpreting regions the policy rarely visits.

In [ ]:
env = gym.make("MountainCar-v0")
low, high = env.observation_space.low, env.observation_space.high
positions = np.linspace(low[0], high[0], 100)
velocities = np.linspace(low[1], high[1], 100)
values = np.empty((len(velocities), len(positions)))

for i, velocity in enumerate(velocities):
    for j, position in enumerate(positions):
        x = polynomial_value_features(np.array([position, velocity]), low, high)
        values[i, j] = prediction_w @ x

plt.figure(figsize=(8, 4))
plt.contourf(positions, velocities, values, levels=30, cmap="viridis")
plt.colorbar(label=r"predicted $\hat v(s,\mathbf w)$")
plt.xlabel("position")
plt.ylabel("velocity")
plt.title("TD prediction for the fixed momentum policy")
plt.show()
env.close()

---
## Part 4 — Control with discretised Q-learning

Control asks: **how should the agent act?**

A Q-table needs discrete state indices. We can create them by dividing position and velocity into bins. With `n_position_bins × n_velocity_bins` bins, the table has shape

```text
(n_position_bins, n_velocity_bins, n_actions)
```

Every continuous state in the same rectangle shares one table entry. This is simple, but creates sharp boundaries and learns nothing about neighbouring bins.

### 4.1 Map a continuous state to a bin

Complete `discretise_state`. Scale the state into the interval $[0,1]$, multiply by the number of bins, convert to integers, and clip the result. Clipping is needed because the upper bound would otherwise produce an out-of-range index.

In [ ]:
def discretise_state(
    state: np.ndarray,
    low: np.ndarray,
    high: np.ndarray,
    bins: tuple[int, int],
) -> tuple[int, int]:
    # TODO: scale, bin, and clip both state components.
    scaled = ...
    indices = ...
    indices = ...
    return int(indices[0]), int(indices[1])


# Sanity checks (run after completing the function).
env = gym.make("MountainCar-v0")
low, high = env.observation_space.low, env.observation_space.high
assert discretise_state(low, low, high, (18, 14)) == (0, 0)
assert discretise_state(high, low, high, (18, 14)) == (17, 13)
print("Example bin:", discretise_state(np.array([-0.5, 0.0]), low, high, (18, 14)))
env.close()

In [ ]:
def epsilon_greedy(values: np.ndarray, epsilon: float, rng: np.random.Generator) -> int:
    """Choose randomly with probability epsilon; otherwise choose a greedy action."""
    if rng.random() < epsilon:
        return int(rng.integers(len(values)))
    # Random tie-breaking avoids a systematic preference for action 0.
    best = np.flatnonzero(values == values.max())
    return int(rng.choice(best))


@dataclass
class ControlStats:
    returns: np.ndarray
    successes: np.ndarray

### 4.2 Implement discretised Q-learning

Q-learning uses the off-policy target

$$R_{t+1}+\gamma\max_{a'}Q(S_{t+1},a').$$

Complete the target and update. Do not bootstrap from a terminal or time-limit transition.

In [ ]:
def train_discretised_q_learning(
    episodes: int = 5000,
    bins: tuple[int, int] = (18, 14),
    alpha: float = 0.15,
    gamma: float = 1.0,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.02,
    seed: int = SEED,
):
    env = gym.make("MountainCar-v0")
    low, high = env.observation_space.low, env.observation_space.high
    n_actions = env.action_space.n
    Q = np.zeros((*bins, n_actions), dtype=np.float64)
    rng = np.random.default_rng(seed)
    returns, successes = [], []

    for episode in range(episodes):
        state, info = env.reset(seed=seed + episode)
        discrete_state = discretise_state(state, low, high, bins)
        fraction = episode / max(episodes - 1, 1)
        epsilon = epsilon_start + fraction * (epsilon_end - epsilon_start)
        episode_return = 0.0

        while True:
            action = epsilon_greedy(Q[discrete_state], epsilon, rng)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            next_discrete_state = discretise_state(next_state, low, high, bins)

            # TODO 1: form the Q-learning target (no bootstrap when done).
            target = ...
            # TODO 2: update Q(discrete_state, action).
            Q[discrete_state + (action,)] += ...

            episode_return += reward
            discrete_state = next_discrete_state
            if done:
                break

        returns.append(episode_return)
        successes.append(float(terminated))

    env.close()
    return Q, ControlStats(np.asarray(returns), np.asarray(successes))

In [ ]:
# This may take a minute.
Q_discrete, discrete_stats = train_discretised_q_learning()
print("Final 100-episode mean return:", discrete_stats.returns[-100:].mean())
print("Final 100-episode success rate:", discrete_stats.successes[-100:].mean())

---
## Part 5 — Control with linear SARSA

Rather than index a table, approximate each action value:

$$\hat q(s,a,\mathbf W)=\mathbf w_a^\top\mathbf x(s).$$

SARSA is on-policy: the next action $A_{t+1}$ is selected using the same $\varepsilon$-greedy policy that generates experience. Its target is

$$R_{t+1}+\gamma\hat q(S_{t+1},A_{t+1},\mathbf W),$$

and only the weights for action $A_t$ are updated:

$$\mathbf w_{A_t}\leftarrow\mathbf w_{A_t}+\alpha\delta_t\mathbf x(S_t).$$

### 5.1 Begin with simple features

These features contain a bias, scaled position, and scaled velocity. The model can only represent a plane for each action. This strong generalisation may be too simple for the problem—and that is a useful result.

In [ ]:
env = gym.make("MountainCar-v0")
STATE_LOW = env.observation_space.low.copy()
STATE_HIGH = env.observation_space.high.copy()
env.close()


def simple_features(state: np.ndarray) -> np.ndarray:
    position, velocity = scale_state(state, STATE_LOW, STATE_HIGH)
    return np.array([1.0, position, velocity])

### 5.2 Implement linear SARSA

The same trainer will work with any function that turns a state into a feature vector. Complete the four TODOs. Pay particular attention to the order: choose the next action before constructing the SARSA target.

In [ ]:
def train_linear_sarsa(
    feature_fn: Callable[[np.ndarray], np.ndarray],
    episodes: int = 1000,
    alpha: float = 0.02,
    gamma: float = 1.0,
    epsilon_start: float = 0.3,
    epsilon_end: float = 0.01,
    seed: int = SEED,
):
    env = gym.make("MountainCar-v0")
    n_actions = env.action_space.n
    n_features = len(feature_fn(env.observation_space.sample()))
    W = np.zeros((n_actions, n_features), dtype=np.float64)
    rng = np.random.default_rng(seed)
    returns, successes = [], []

    for episode in range(episodes):
        state, info = env.reset(seed=seed + episode)
        x = feature_fn(state)
        fraction = episode / max(episodes - 1, 1)
        epsilon = epsilon_start + fraction * (epsilon_end - epsilon_start)
        action = epsilon_greedy(W @ x, epsilon, rng)
        episode_return = 0.0

        while True:
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            next_x = feature_fn(next_state)

            if done:
                next_action = None
                target = reward
            else:
                # TODO 1: choose next_action from the current behaviour policy.
                next_action = ...
                # TODO 2: form the SARSA target.
                target = ...

            # TODO 3: calculate the TD error for q(state, action).
            delta = ...
            # TODO 4: update only the weights for the action just taken.
            W[action] += ...

            episode_return += reward
            if done:
                break
            state, x, action = next_state, next_x, next_action

        returns.append(episode_return)
        successes.append(float(terminated))

    env.close()
    return W, ControlStats(np.asarray(returns), np.asarray(successes))

In [ ]:
W_simple, simple_stats = train_linear_sarsa(simple_features)
print("Final 100-episode mean return:", simple_stats.returns[-100:].mean())
print("Final 100-episode success rate:", simple_stats.successes[-100:].mean())

---
## Part 6 — A richer representation: tile coding

One grid creates abrupt boundaries. Tile coding overlays several slightly shifted grids. Each state activates one tile in every tiling. Nearby states share some—but not necessarily all—active features.

This gives us:

- **local generalisation:** an update affects a region rather than one exact state;
- **overlapping features:** crossing one boundary does not change every active feature; and
- **a linear approximator:** $\hat q$ is still linear in the weights, even though it can represent a complex surface.

The tile coder is supplied so that the experiment stays focused on prediction, control, and representation rather than indexing details.

In [ ]:
class TileCoder:
    def __init__(
        self,
        low: np.ndarray,
        high: np.ndarray,
        tiles_per_dimension: tuple[int, int] = (8, 8),
        n_tilings: int = 8,
    ):
        self.low = np.asarray(low, dtype=np.float64)
        self.high = np.asarray(high, dtype=np.float64)
        self.tiles = np.asarray(tiles_per_dimension, dtype=int)
        self.n_tilings = n_tilings
        self.features_per_tiling = int(np.prod(self.tiles))
        self.n_features = self.n_tilings * self.features_per_tiling

    def __call__(self, state: np.ndarray) -> np.ndarray:
        scaled = (np.asarray(state) - self.low) / (self.high - self.low)
        features = np.zeros(self.n_features, dtype=np.float64)

        for tiling in range(self.n_tilings):
            # Different shifts in the two dimensions reduce aligned boundaries.
            shift = np.array([tiling, (3 * tiling) % self.n_tilings]) / self.n_tilings
            coordinates = np.floor(scaled * self.tiles + shift).astype(int)
            coordinates = np.clip(coordinates, 0, self.tiles - 1)
            within_tiling = np.ravel_multi_index(tuple(coordinates), tuple(self.tiles))
            features[tiling * self.features_per_tiling + within_tiling] = 1.0

        return features


tile_features = TileCoder(STATE_LOW, STATE_HIGH, tiles_per_dimension=(8, 8), n_tilings=8)
example = tile_features(np.array([-0.5, 0.0]))
print("Number of features:", len(example))
print("Active features per state:", int(example.sum()))

In [ ]:
# A smaller step size is used because every state activates several tiles.
W_tiles, tile_stats = train_linear_sarsa(
    tile_features, episodes=1500, alpha=0.08 / tile_features.n_tilings
)
print("Final 100-episode mean return:", tile_stats.returns[-100:].mean())
print("Final 100-episode success rate:", tile_stats.successes[-100:].mean())

### 6.1 Compare learning

A moving average makes the trend easier to see. Runs may vary even with fixed seeds because learning is sensitive to exploration and hyperparameters.

In [ ]:
def moving_average(values: np.ndarray, window: int = 100):
    if len(values) < window:
        return np.arange(len(values)), values
    smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
    return np.arange(window - 1, len(values)), smoothed


plt.figure(figsize=(10, 5))
for label, stats in [
    ("discretised Q-learning", discrete_stats),
    ("linear SARSA: simple features", simple_stats),
    ("linear SARSA: tile coding", tile_stats),
]:
    x_axis, mean_return = moving_average(stats.returns)
    plt.plot(x_axis, mean_return, label=label)

plt.xlabel("episode")
plt.ylabel("return (100-episode moving average)")
plt.title("How representation changes learning")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### 6.2 Evaluate greedy policies

Training return includes exploratory actions, so evaluate each final policy separately with $\varepsilon=0$.

In [ ]:
def evaluate_policy(action_fn: Callable[[np.ndarray], int], episodes: int = 100, seed: int = 10_000):
    env = gym.make("MountainCar-v0")
    returns, successes = [], []

    for episode in range(episodes):
        state, info = env.reset(seed=seed + episode)
        episode_return = 0.0
        while True:
            state, reward, terminated, truncated, info = env.step(action_fn(state))
            episode_return += reward
            if terminated or truncated:
                break
        returns.append(episode_return)
        successes.append(float(terminated))

    env.close()
    return np.mean(returns), np.mean(successes)


bins = Q_discrete.shape[:2]
discrete_policy = lambda s: int(np.argmax(Q_discrete[discretise_state(s, STATE_LOW, STATE_HIGH, bins)]))
simple_policy = lambda s: int(np.argmax(W_simple @ simple_features(s)))
tile_policy = lambda s: int(np.argmax(W_tiles @ tile_features(s)))

for name, policy in [
    ("discretised Q-learning", discrete_policy),
    ("linear SARSA: simple features", simple_policy),
    ("linear SARSA: tile coding", tile_policy),
]:
    mean_return, success_rate = evaluate_policy(policy)
    print(f"{name:38s} mean return={mean_return:7.1f}  success={success_rate:5.1%}")

### 6.3 Visualise the learned cost-to-go

For Mountain Car it is conventional to display $-\max_a\hat q(s,a)$, an estimate of the remaining cost. A useful representation should be able to form a non-trivial surface over position and velocity.

In [ ]:
def plot_cost_to_go(W: np.ndarray, feature_fn: Callable[[np.ndarray], np.ndarray], title: str):
    positions = np.linspace(STATE_LOW[0], STATE_HIGH[0], 80)
    velocities = np.linspace(STATE_LOW[1], STATE_HIGH[1], 80)
    costs = np.empty((len(velocities), len(positions)))
    for i, velocity in enumerate(velocities):
        for j, position in enumerate(positions):
            q_values = W @ feature_fn(np.array([position, velocity]))
            costs[i, j] = -q_values.max()

    plt.figure(figsize=(7, 4))
    plt.contourf(positions, velocities, costs, levels=30, cmap="magma")
    plt.colorbar(label="estimated cost-to-go")
    plt.xlabel("position")
    plt.ylabel("velocity")
    plt.title(title)
    plt.show()


plot_cost_to_go(W_simple, simple_features, "Simple linear features")
plot_cost_to_go(W_tiles, tile_features, "Tile-coded features")

---
## Part 7 — Questions (write brief answers below)

1. **Prediction vs control:** What was held fixed in Part 3, and what changed when we moved to Parts 4–6?
2. **Bootstrapping:** Identify the bootstrapped quantity in TD(0), Q-learning, and SARSA. Why is there no bootstrap term at the end of an episode?
3. **Discretisation:** What happens if you use very few bins? What new problem appears if you use a very large number of bins? Try at least two settings.
4. **On-policy vs off-policy:** Which target makes Q-learning off-policy? Why is the SARSA target on-policy?
5. **Generalisation:** When one transition updates the tile-coded agent, which other states can be affected? How does this differ from a Q-table?
6. **Representation:** Compare the learning curves and cost-to-go plots for simple features and tile coding. What can tile coding represent that the simple model cannot?
7. **Step size:** Why is the tile-coded step size divided by the number of tilings?
8. **Conclusion:** Which approach would you choose for Mountain Car, and what evidence from your experiments supports your choice?

### Answers

1. 

2. 

3. 

4. 

5. 

6. 

7. 

8. 

---
## Optional extensions

Choose one:

1. **Expected SARSA:** replace the sampled next-action value with its expectation under the $\varepsilon$-greedy policy.
2. **Feature study:** vary the number of tilings and tiles per dimension. Compare performance and the number of weights.
3. **Algorithm comparison:** implement semi-gradient Q-learning using the same tile-coded representation.
4. **Policy map:** colour each point in the position–velocity plane by its greedy action. Explain the resulting strategy.
5. **Repeated trials:** run at least five seeds and plot a mean curve with uncertainty rather than drawing conclusions from one run.